# Statistical Arbitrage Strategy Demonstration

This notebook recreates `qc_projects/main_statistical_arbitrage.py`, where we:
- Track a correlated basket: AAPL, MSFT, GOOGL
- Compute rolling z-scores per stock over a 20-day window
- Compare each stock's z-score vs basket mean z-score
- Enter long/short on large deviations and exit on convergence

## Step 1: Imports and Parameters

Parameters match the Lean algorithm defaults: 20-day rolling window, entry at ±1.5 deviation, and exit at ±0.3.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option('display.max_columns', None)

symbols = ['AAPL', 'MSFT', 'GOOGL']
start_date = '2020-01-01'
end_date = '2022-01-01'
window = 20
entry_threshold = 1.5
exit_threshold = 0.3
position_size = 0.3
warmup_period = window + 5

print(f'Symbols: {symbols}')
print(f'Window={window}, Entry=±{entry_threshold}, Exit=±{exit_threshold}, Position={position_size:.1f}')
print(f'Backtest: {start_date} → {end_date}')

## Step 2: Download Daily Data for Basket

We fetch daily closes for each symbol and align all dates to a common index.

In [ ]:
close_map = {}
for sym in symbols:
    df = yf.download(
        sym,
        start=start_date,
        end=end_date,
        interval='1d',
        progress=False,
        multi_level_index=False,
        auto_adjust=False,
    )
    if df.empty:
        raise RuntimeError(f'No OHLCV data for {sym}')
    close_map[sym] = df['Close']

prices = pd.DataFrame(close_map).dropna().copy()
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.index.name = 'date'

print(f'Aligned rows: {len(prices):,}')
prices.head()

## Step 3: Compute Rolling Z-Scores and Basket Deviation

For each symbol: `z = (price - rolling_mean) / rolling_std`. Then we compute `basket_mean` and each symbol's deviation from that mean.

In [ ]:
features = prices.copy()
for sym in symbols:
    roll_mean = features[sym].rolling(window).mean()
    roll_std = features[sym].rolling(window).std(ddof=0)
    z_col = f'z_{sym}'
    features[z_col] = (features[sym] - roll_mean) / roll_std
    features[z_col] = features[z_col].replace([np.inf, -np.inf], np.nan)

z_cols = [f'z_{s}' for s in symbols]
features = features.dropna(subset=z_cols).copy()
features['basket_mean'] = features[z_cols].mean(axis=1)

for sym in symbols:
    features[f'dev_{sym}'] = features[f'z_{sym}'] - features['basket_mean']

if len(features) > warmup_period:
    features = features.iloc[warmup_period:].copy()

features[[*z_cols, 'basket_mean', 'dev_AAPL', 'dev_MSFT', 'dev_GOOGL']].head()

## Step 4: Replay QC Trading Logic

Per symbol:
- Enter long when deviation < -1.5
- Enter short when deviation > +1.5
- Exit when |deviation| < 0.3

In [ ]:
positions = {}
trades = []
signal_rows = []

for ts, row in features.iterrows():
    for sym in symbols:
        dev = float(row[f'dev_{sym}'])
        px = float(row[sym])
        pos = positions.get(sym)

        if pos is None:
            if dev < -entry_threshold:
                positions[sym] = {'side': 'long', 'entry_time': ts, 'entry_px': px, 'entry_dev': dev}
                signal_rows.append({'timestamp': ts, 'symbol': sym, 'type': 'entry', 'side': 'long', 'deviation': dev})
            elif dev > entry_threshold:
                positions[sym] = {'side': 'short', 'entry_time': ts, 'entry_px': px, 'entry_dev': dev}
                signal_rows.append({'timestamp': ts, 'symbol': sym, 'type': 'entry', 'side': 'short', 'deviation': dev})
        else:
            if abs(dev) < exit_threshold:
                raw_ret = px / pos['entry_px'] - 1.0
                signed_ret = raw_ret if pos['side'] == 'long' else -raw_ret
                weighted_ret_pct = signed_ret * position_size * 100
                trades.append({
                    'symbol': sym,
                    'side': pos['side'],
                    'entry_time': pos['entry_time'],
                    'exit_time': ts,
                    'entry_price': round(pos['entry_px'], 2),
                    'exit_price': round(px, 2),
                    'entry_deviation': round(pos['entry_dev'], 4),
                    'exit_deviation': round(dev, 4),
                    'return_pct': round(weighted_ret_pct, 3),
                })
                signal_rows.append({'timestamp': ts, 'symbol': sym, 'type': 'exit', 'side': pos['side'], 'deviation': dev})
                positions.pop(sym, None)

trades_df = pd.DataFrame(trades)
signals_df = pd.DataFrame(signal_rows)
print(f'Trades closed: {len(trades_df)} | Open positions: {len(positions)}')
trades_df.head() if not trades_df.empty else 'No trades generated'

## Step 5: Performance Summary

Quick trade-level metrics to validate if thresholds are producing positive expectancy.

In [ ]:
if trades_df.empty:
    summary = {
        'total_trades': 0,
        'win_rate_pct': 0.0,
        'avg_return_pct': 0.0,
        'median_return_pct': 0.0,
        'best_trade_pct': 0.0,
        'worst_trade_pct': 0.0,
    }
else:
    summary = {
        'total_trades': int(len(trades_df)),
        'win_rate_pct': round((trades_df['return_pct'] > 0).mean() * 100, 1),
        'avg_return_pct': round(float(trades_df['return_pct'].mean()), 3),
        'median_return_pct': round(float(trades_df['return_pct'].median()), 3),
        'best_trade_pct': round(float(trades_df['return_pct'].max()), 3),
        'worst_trade_pct': round(float(trades_df['return_pct'].min()), 3),
    }

summary

## Step 6: Visualize Deviations and Trade Signals

We plot each symbol's deviation from basket mean and mark entries/exits.

In [ ]:
graphs_dir = Path('graphs')
graphs_dir.mkdir(exist_ok=True)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=('Basket Price Context', 'Deviation vs Basket Mean'),
)

for sym, color in [('AAPL', '#1f77b4'), ('MSFT', '#ff7f0e'), ('GOOGL', '#2ca02c')]:
    fig.add_trace(go.Scatter(x=features.index, y=features[sym], name=f'{sym} Price', line=dict(color=color)), row=1, col=1)

for sym, color in [('AAPL', '#1f77b4'), ('MSFT', '#ff7f0e'), ('GOOGL', '#2ca02c')]:
    fig.add_trace(go.Scatter(x=features.index, y=features[f'dev_{sym}'], name=f'{sym} Deviation', line=dict(color=color)), row=2, col=1)

fig.add_hline(y=entry_threshold, line=dict(color='#d62728', dash='dash'), row=2, col=1)
fig.add_hline(y=-entry_threshold, line=dict(color='#d62728', dash='dash'), row=2, col=1)
fig.add_hline(y=exit_threshold, line=dict(color='#9467bd', dash='dot'), row=2, col=1)
fig.add_hline(y=-exit_threshold, line=dict(color='#9467bd', dash='dot'), row=2, col=1)
fig.add_hline(y=0, line=dict(color='#7f7f7f', dash='dot'), row=2, col=1)

if not signals_df.empty:
    entries = signals_df[signals_df['type'] == 'entry']
    exits = signals_df[signals_df['type'] == 'exit']
    for sym in symbols:
        ent = entries[entries['symbol'] == sym]
        ex = exits[exits['symbol'] == sym]
        if not ent.empty:
            fig.add_trace(
                go.Scatter(
                    x=ent['timestamp'],
                    y=ent['deviation'],
                    mode='markers',
                    marker=dict(symbol='triangle-up', size=8, color='#2ca02c'),
                    name=f'{sym} Entries',
                ),
                row=2, col=1
            )
        if not ex.empty:
            fig.add_trace(
                go.Scatter(
                    x=ex['timestamp'],
                    y=ex['deviation'],
                    mode='markers',
                    marker=dict(symbol='triangle-down', size=8, color='#d62728'),
                    name=f'{sym} Exits',
                ),
                row=2, col=1
            )

fig.update_layout(height=780, title='Statistical Arbitrage Basket Deviations')
output_path = graphs_dir / 'statistical_arbitrage_signals.html'
fig.write_html(output_path)
fig.show()
print(f'Saved interactive chart to {output_path}')

## Step 7: Trade Log and Recommendation

Inspect recent trades and produce a quick recommendation based on aggregate stats.

In [ ]:
if not trades_df.empty:
    display(trades_df.tail(15))

if summary['total_trades'] == 0:
    rec = 'HOLD - no strong divergences under current parameters.'
elif summary['win_rate_pct'] >= 55 and summary['avg_return_pct'] > 0:
    rec = 'BUY (strategy-ready) - convergence behavior appears favorable.'
elif summary['avg_return_pct'] <= 0:
    rec = 'AVOID/REFINE - tune thresholds/window before deployment.'
else:
    rec = 'NEUTRAL - mixed outcomes; continue calibration.'

print('Recommendation:', rec)

## Recap

- This demo mirrors the QC state machine for basket-level statistical arbitrage.
- Use it to tune `window`, entry/exit thresholds, and basket composition before syncing changes back to Lean.
- Saved chart: `graphs/statistical_arbitrage_signals.html`.